In [2]:
# Run cells selectively according to your needs.

from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

In [ ]:
dataset = load_dataset('roneneldan/TinyStories', split='train')

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"])
token_ids = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
tokenized_tinystories_path = '../datasets/tokenized_tinystories'
token_ids.save_to_disk(tokenized_tinystories_path)

In [ ]:
token_ids = load_from_disk(tokenized_tinystories_path)

In [4]:
chunked_tokenized_tinystories_path = '../datasets/chunked_tokenized_tinystories'

In [ ]:
eos_id= tokenizer.eos_token_id
block_size = 2048

def chunk_gen():
  buffer = []
  for batch in token_ids.iter(batch_size=1000):
    for ids in batch['input_ids']:
      buffer.extend(ids)
      buffer.append(eos_id)

      while(len(buffer) >= block_size):
        chunk = buffer[:block_size]
        buffer = buffer[block_size:]

        yield {'input_ids': chunk}

from datasets import Dataset
chunked_dataset = Dataset.from_generator(chunk_gen)
chunked_dataset.save_to_disk(chunked_tokenized_tinystories_path)


In [5]:
chunked_dataset = load_from_disk(chunked_tokenized_tinystories_path)

In [10]:
# list dataset splits
chunked_dataset.column_names

['input_ids']